# HateGuard - Deploy natively on Google Colab

This notebook acts as an automated setup to run the HateGuard product.
It uses Google Colab's builtin secure proxy. Once started, click the link to open the app!

In [ ]:
import os
# 1. Clean previous runs and securely clone the latest repository
!rm -rf hate-comment-dectection
!git clone https://github.com/ATANU28-bit/hate-comment-dectection.git
%cd hate-comment-dectection

In [ ]:
# 2. Install dependencies
!sudo apt update && sudo apt install ffmpeg -y
!pip install -r requirements.txt

# Install UI dependencies
%cd ui
!npm install
%cd ..

In [ ]:
import subprocess
import time
from google.colab import output

print("Starting Backend on port 8000...")
backend = subprocess.Popen(["uvicorn", "src.api:app", "--host", "127.0.0.1", "--port", "8000"])

# Set UI to route queries via Vite's proxy path "/api"
with open("ui/.env", "w", encoding="utf-8") as f:
    f.write("VITE_API_URL=/api\n")

print("Starting Frontend Vite proxy on port 5173...")
frontend = subprocess.Popen(["npm", "run", "dev", "--prefix", "ui", "--", "--host", "127.0.0.1", "--port", "5173"])

time.sleep(5)

print("\n=========================================================")
print("🚀 YOUR APPLICATION IS READY!")
print("=========================================================")

# Use Colab's native secure port mapping instead of unreliable tunnels
output.serve_kernel_port_as_window(5173)

print("\nHold this cell running! Press stop to shut down.")
try:
    frontend.wait()
except KeyboardInterrupt:
    print("Shutting down servers...")
    backend.terminate()
    frontend.terminate()
